In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json
from typing import Optional, Union
import os


def visualize_json(
    data: Union[str, dict, pd.DataFrame],
    x: Optional[str] = None,
    y: Optional[str] = None,
    color: Optional[str] = None,
    plot_type: str = "auto",
    title: Optional[str] = None,
    normalize_json: bool = False,
    export_path: Optional[str] = None,
    dropna: bool = True,
    **kwargs
) -> None:
    """
    Automatically visualize JSON/DataFrame data with Plotly using optimal plot type.
    Automatically converts epoch timestamps to datetime.

    Parameters:
    - data: JSON file path, JSON string, dict, or DataFrame
    - x: X-axis column (auto-detected if None)
    - y: Y-axis column (auto-detected if None)
    - color: Color grouping column
    - plot_type: 'auto' (default) or specific type ('scatter', 'line', 'bar', 'histogram', 'box', 'pie', 'scatter_matrix')
    - title: Custom plot title
    - normalize_json: Flatten nested JSON using pd.json_normalize()
    - export_path: File path to export the plot (HTML format)
    - dropna: Whether to drop missing values (default=True)
    - **kwargs: Additional Plotly Express parameters

    Example:
    >>> visualize_json('data.json')
    >>> visualize_json({"date": ["2021-01-01", "2021-01-02"], "value": [10, 15]})
    """
    try:
        # Load and parse data
        df = _load_data(data, normalize_json)

        # Convert epoch columns to datetime
        df = _convert_epoch_to_datetime(df)

        # Auto-detect columns if not specified
        x, y = _auto_detect_columns(df, x, y)

        # Validate columns exist
        _validate_columns(df, x, y, color)

        # Handle missing values
        if dropna:
            df = df.dropna(subset=[col for col in [x, y, color] if col])

        # Automatically determine the best plot type if not specified
        if plot_type == "auto":
            plot_type = _infer_plot_type(df, x, y)

        # Generate and display the plot
        fig = _create_plot(df, x, y, color, plot_type, title, **kwargs)

        # Export the plot if export path is provided
        if export_path:
            fig.write_html(export_path)
            print(f"Plot exported to {export_path}")

        # Display the plot
        fig.show()

    except Exception as e:
        print(f"Error: {str(e)}")


def _load_data(data: Union[str, dict, pd.DataFrame], normalize: bool) -> pd.DataFrame:
    """Load JSON or DataFrame data into a Pandas DataFrame."""
    if isinstance(data, pd.DataFrame):
        return data.copy()

    if isinstance(data, str):
        if os.path.isfile(data):
            with open(data, 'r') as f:
                json_data = json.load(f)
        else:
            json_data = json.loads(data)
    elif isinstance(data, dict):
        json_data = data
    else:
        raise ValueError("Input must be DataFrame, JSON string, file path, or dict")

    return pd.json_normalize(json_data) if normalize else pd.DataFrame(json_data)


def _convert_epoch_to_datetime(df: pd.DataFrame) -> pd.DataFrame:
    """Automatically convert epoch columns to datetime format."""
    for col in df.columns:
        if pd.api.types.is_integer_dtype(df[col]):
            # Detect epoch format: Milliseconds (13 digits) or Seconds (10 digits)
            if df[col].max() > 1e12:  # Milliseconds
                df[col] = pd.to_datetime(df[col], unit='ms', utc=True)
                print(f"Converted {col} from epoch (ms) to datetime.")
            elif df[col].max() > 1e9:  # Seconds
                df[col] = pd.to_datetime(df[col], unit='s', utc=True)
                print(f"Converted {col} from epoch (s) to datetime.")
    
    return df


def _auto_detect_columns(df: pd.DataFrame, x: Optional[str], y: Optional[str]) -> tuple:
    """Automatically detect x and y columns."""
    if x is None:
        x = df.columns[0] if len(df.columns) > 0 else None

    if y is None and len(df.columns) > 1:
        numeric_cols = df.select_dtypes(include="number").columns
        y = numeric_cols[0] if len(numeric_cols) > 0 else df.columns[1]

    return x, y


def _validate_columns(df: pd.DataFrame, x: str, y: Optional[str], color: Optional[str]) -> None:
    """Ensure selected columns exist in the DataFrame."""
    missing = [col for col in [x, y, color] if col and col not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")


def _infer_plot_type(df: pd.DataFrame, x: str, y: Optional[str]) -> str:
    """Intelligently determine the best plot type."""
    
    # One column: Histogram or Pie
    if y is None:
        if pd.api.types.is_numeric_dtype(df[x]):
            return "histogram"
        else:
            return "pie" if df[x].nunique() < 10 else "bar"

    # X-axis is datetime: Line plot
    if pd.api.types.is_datetime64_any_dtype(df[x]):
        return "line"

    # Two numeric columns: Scatter plot
    if pd.api.types.is_numeric_dtype(df[x]) and pd.api.types.is_numeric_dtype(df[y]):
        return "scatter"

    # Categorical + Numeric: Bar or Box plot
    if not pd.api.types.is_numeric_dtype(df[x]) and pd.api.types.is_numeric_dtype(df[y]):
        return "box" if df[x].nunique() > 5 else "bar"

    # Two categorical columns: Pie or Bar
    if not pd.api.types.is_numeric_dtype(df[x]) and not pd.api.types.is_numeric_dtype(df[y]):
        return "pie" if df[x].nunique() < 10 else "bar"

    # More than 2 dimensions: Scatter matrix
    if len(df.columns) > 2:
        return "scatter_matrix"

    return "scatter"


def _create_plot(
    df: pd.DataFrame, x: str, y: Optional[str], color: Optional[str],
    plot_type: str, title: Optional[str], **kwargs
) -> go.Figure:
    """Generate Plotly visualization."""
    plot_map = {
        "scatter": px.scatter,
        "line": px.line,
        "bar": px.bar,
        "histogram": px.histogram,
        "box": px.box,
        "pie": px.pie,
        "scatter_matrix": px.scatter_matrix
    }

    if plot_type == "histogram":
        fig = px.histogram(df, x=x, color=color, title=title, **kwargs)
    elif plot_type == "pie":
        fig = px.pie(df, names=x, values=y, title=title, **kwargs)
    elif plot_type == "scatter_matrix":
        fig = px.scatter_matrix(df, dimensions=[x, y] if y else [x], title=title, **kwargs)
    else:
        fig = plot_map[plot_type](
            df, x=x, y=y, color=color,
            title=title or f"{plot_type.title()} Plot: {x}" + (f" vs {y}" if y else ""),
            **kwargs
        )

    return fig


In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json
from typing import Optional, Union
import os


def visualize_json(
    data: Union[str, dict, pd.DataFrame],
    x: Optional[str] = None,
    y: Optional[str] = None,
    color: Optional[str] = None,
    plot_type: str = "auto",
    title: Optional[str] = None,
    normalize_json: bool = False,
    export_path: Optional[str] = None,
    dropna: bool = True,
    **kwargs
) -> None:
    """
    Automatically visualize JSON/DataFrame data with Plotly using optimal plot type.
    Automatically converts epoch timestamps to datetime.

    Parameters:
    - data: JSON file path, JSON string, dict, or DataFrame
    - x: X-axis column (auto-detected if None)
    - y: Y-axis column (auto-detected if None)
    - color: Color grouping column
    - plot_type: 'auto' (default) or specific type ('scatter', 'line', 'bar', 'histogram', 'box', 'pie', 'scatter_matrix')
    - title: Custom plot title
    - normalize_json: Flatten nested JSON using pd.json_normalize()
    - export_path: File path to export the plot (HTML format)
    - dropna: Whether to drop missing values (default=True)
    - **kwargs: Additional Plotly Express parameters

    Example:
    >>> visualize_json('data.json')
    >>> visualize_json({"date": ["2021-01-01", "2021-01-02"], "value": [10, 15]})
    """
    try:
        # Load and parse data
        df = _load_data(data, normalize_json)

        # Convert epoch columns to datetime
        df = _convert_epoch_to_datetime(df)

        # Auto-detect columns if not specified
        x, y = _auto_detect_columns(df, x, y)

        # Validate columns exist
        _validate_columns(df, x, y, color)

        # Handle missing values
        if dropna:
            df = df.dropna(subset=[col for col in [x, y, color] if col])

        # Automatically determine the best plot type if not specified
        if plot_type == "auto":
            plot_type = _infer_plot_type(df, x, y)

        # Generate and display the plot
        fig = _create_plot(df, x, y, color, plot_type, title, **kwargs)

        # Export the plot if export path is provided
        if export_path:
            fig.write_html(export_path)
            print(f"Plot exported to {export_path}")

        # Display the plot
        fig.show()

    except Exception as e:
        print(f"Error: {str(e)}")